<div dir="rtl">

# الدرس 27: الإفراط في التعلّم وقلّة التعلّم

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

## منحنى التعقيد، بالكود

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 60)
y = np.sin(2.6 * x) * 1.2 + 0.4 * x + rng.normal(0, 0.28, x.size)
X = x.reshape(-1, 1)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.4, random_state=0)

degrees = range(0, 16)
tr_err, val_err = [], []
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), StandardScaler(), LinearRegression()).fit(X_tr, y_tr)
    tr_err.append(mean_squared_error(y_tr, m.predict(X_tr)))
    val_err.append(mean_squared_error(y_val, m.predict(X_val)))

plt.figure(figsize=(6, 3.4))
plt.plot(degrees, tr_err, "o-", lw=2, label="train")
plt.plot(degrees, val_err, "o-", lw=2, label="validation")
plt.yscale("log")
plt.xlabel("polynomial degree (model complexity)"); plt.ylabel("MSE (log scale)")
plt.legend(); plt.grid(True)
plt.show()

best = int(np.argmin(val_err))
print("best degree by validation:", best, " train MSE:", round(tr_err[best], 3), " val MSE:", round(val_err[best], 3))

<div dir="rtl">

## منحنى التعلّم: هل تحتاج بيانات أكثر؟

</div>

In [ ]:
from sklearn.model_selection import learning_curve
from sklearn.datasets import load_digits
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

Xd, yd = load_digits(return_X_y=True)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (name, model) in zip(axes, [("deep tree", DecisionTreeClassifier(random_state=0)),
                                     ("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)))]):
    sizes, tr, va = learning_curve(model, Xd, yd, cv=5, train_sizes=np.linspace(0.1, 1.0, 6))
    ax.plot(sizes, tr.mean(axis=1), "o-", lw=2, label="train accuracy")
    ax.plot(sizes, va.mean(axis=1), "o-", lw=2, label="validation accuracy")
    ax.set_title(name); ax.set_xlabel("training examples"); ax.grid(True)
axes[0].set_ylabel("accuracy"); axes[0].legend()
plt.tight_layout()
plt.show()

<div dir="rtl">

## تمرين: اختر العمق بالتحقق

استخدم بيانات أورام الثدي. قسّمها: 60% تدريب، و 20% تحقق، و 20% اختبار.

1. درّب أشجار قرار بأعماق من 1 إلى 12. لكل عمق، احسب الدقة على التدريب والتحقق.
2. ارسم الخطين. أين تبدأ الفجوة بالكبر؟
3. اختر العمق الأفضل حسب التحقق، ثم قيّمه مرة واحدة على الاختبار.

</div>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier

Xb, yb = load_breast_cancer(return_X_y=True)
# قسّم إلى ثلاث مجموعات، ثم جرّب الأعماق